In [ ]:
import os
import onnx
import numpy as np
from aimet_onnx import QuantizationSimModel
from aimet_onnx.common.defs import QuantScheme
import aimet_onnx

# ----------------------------
# 1. shape固定済み ONNX をロード
# ----------------------------
#model = onnx.load("/root/AIMET/convert/simplified_fix.onnx")
model = onnx.load("/root/AIMET/bge-onnx/model.onnx")  # ← 適宜変更

OUT_DIR="/root/AIMET_ENV/tools/bge"  # ← 適宜変更
# 2) (推奨) simplify
try:
    import onnxsim
    model, _ = onnxsim.simplify(model)
except Exception as e:
    print("onnxsim simplify failed, continue with original model:", repr(e))

# ----------------------------
# 2. QuantizationSimModel 作成
# ----------------------------
providers=["CPUExecutionProvider"]
sim = QuantizationSimModel(

    model,
    param_type=aimet_onnx.int8,
    activation_type=aimet_onnx.int8,
    #quant_scheme=QuantScheme.min_max,
    quant_scheme=QuantScheme.post_training_tf_enhanced,
    #config_file="default",
    config_file="htp_v73",
    #config_file="/root/AIMET_ruri_v2/tool/quantsim_fp_escape.json",  # ← 適宜変更
    providers=providers,
)

# ONNX input names (must match tokenizer outputs)
onnx_input_names = [i.name for i in sim.model.model.graph.input]
print("ONNX inputs:", onnx_input_names)



def make_feed(text: str, max_length: int = 512):
    enc = tokenizer(
        text,
        padding="max_length",
        truncation=True,
        max_length=max_length,
        return_tensors="np",
    )

    feed = {}

    # input_ids / attention_mask / token_type_ids
    if "input_ids" in onnx_input_names:
        feed["input_ids"] = enc["input_ids"].astype(np.int64)

    if "attention_mask" in onnx_input_names:
        feed["attention_mask"] = enc["attention_mask"].astype(np.int64)

    if "token_type_ids" in onnx_input_names:
        if "token_type_ids" in enc:
            feed["token_type_ids"] = enc["token_type_ids"].astype(np.int64)
        else:
            # tokenizer が返さない場合は 0 埋め
            feed["token_type_ids"] = np.zeros_like(
                enc["input_ids"], dtype=np.int64
            )

    # position_ids を明示的に追加
    if "position_ids" in onnx_input_names:
        batch_size, seq_len = enc["input_ids"].shape
        feed["position_ids"] = np.tile(
            np.arange(seq_len, dtype=np.int64),
            (batch_size, 1),
        )

    # 念のためチェック
    missing = [n for n in onnx_input_names if n not in feed]
    if missing:
        raise RuntimeError(
            f"Missing ONNX inputs {missing}. tokenizer keys={list(enc.keys())}"
        )

    return feed

def calib_generator(texts, num_batches: int = 500):
    # aimet_onnx の compute_encodings は iterable を受け取れる（dictをyield）
    for i, t in enumerate(texts):
        if i >= num_batches:
            break
        yield make_feed(t)



calib_texts = [
    "What is semantic search?",
    "Explain retrieval augmented generation.",
    "How does an embedding model work?",
]*200

# tokenizer は事前にロード済み想定
from transformers import AutoTokenizer 
tokenizer = AutoTokenizer.from_pretrained("BAAI/bge-large-en-v1.5")


# ----------------------------
# 5. PTQ Step1 実行（これが目的）
# ----------------------------
sim.compute_encodings(
    calib_generator(calib_texts)
)

print("✅ PTQ Step1 (compute_encodings) completed")

# 6) export (QDQ onnx + encodings json)
sim.export(
    path=OUT_DIR,
    filename_prefix="ruri_normal",
    export_model=True,
    export_int32_bias=True,    # 迷ったらTrueでOK（INT32 bias encodingを生成）
    encoding_version="1.0.0",  # デフォルトでも可
)

print("Done. Exported to:", OUT_DIR)
print("QDQ model:", os.path.join(OUT_DIR, "ruri_normal.onnx"))
print("Encodings:", os.path.join(OUT_DIR, "ruri_normal.encodings"))


2026-04-20 01:47:07,440 - Quant - INFO - Selecting DefaultOpInstanceConfigGenerator to compute the specialized config. hw_version:V73


/root/mteb_AIMET/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


✅ PTQ Step1 (compute_encodings) completed


In [14]:
import inspect
import numpy as np

from mteb.evaluate import evaluate
import mteb.tasks.instruction_retrieval.eng.ifir_scifact_retrieval as scifact_mod

# 1) IFIR SciFact の task クラスを自動で拾う（AbsTask派生）
task_cls = None
picked_name = None
for name, obj in vars(scifact_mod).items():
    if inspect.isclass(obj):
        if any(c.__name__ == "AbsTask" for c in obj.mro()):
            task_cls = obj
            picked_name = name
            break

if task_cls is None:
    raise RuntimeError("No AbsTask-derived class found in IFIR SciFact module")

task = task_cls()
print("Using task:", picked_name)

# 2) MTEBが呼ぶ encode() を持つモデル（sim.session を使う）
class AimetMtebModel:
    def __init__(self, sim, tokenizer, seq_len=512, batch_size=32, output_index=None, normalize=True):
        self.sim = sim
        self.tokenizer = tokenizer
        self.seq_len = seq_len
        self.batch_size = batch_size
        self.output_index = output_index
        self.normalize = normalize

        self.input_names = [i.name for i in sim.session.get_inputs()]
        self.output_names = [o.name for o in sim.session.get_outputs()]
        print("ONNX inputs:", self.input_names)
        print("ONNX outputs:", self.output_names)

    def _make_feed(self, texts):
        if isinstance(texts, str):
            texts = [texts]
        enc = self.tokenizer(
            texts,
            max_length=self.seq_len,
            padding="max_length",
            truncation=True,
            return_tensors="np",
        )

        feed = {}
        if "input_ids" in self.input_names:
            feed["input_ids"] = enc["input_ids"].astype(np.int64)
        if "attention_mask" in self.input_names:
            feed["attention_mask"] = enc["attention_mask"].astype(np.int64)

        if "token_type_ids" in self.input_names:
            if "token_type_ids" in enc:
                feed["token_type_ids"] = enc["token_type_ids"].astype(np.int64)
            else:
                feed["token_type_ids"] = np.zeros_like(enc["input_ids"], dtype=np.int64)

        if "position_ids" in self.input_names:
            bsz, seqlen = feed["input_ids"].shape
            feed["position_ids"] = np.tile(np.arange(seqlen, dtype=np.int64), (bsz, 1))

        missing = [n for n in self.input_names if n not in feed]
        if missing:
            raise RuntimeError(f"Missing ONNX inputs: {missing}. tokenizer keys={list(enc.keys())}")
        return feed

    def encode(self, sentences, batch_size=None, normalize_embeddings=None, **kwargs):
        bs = batch_size if batch_size is not None else self.batch_size
        norm = normalize_embeddings if normalize_embeddings is not None else self.normalize

        embs = []
        for start in range(0, len(sentences), bs):
            batch = sentences[start:start+bs]
            outs = self.sim.session.run(None, self._make_feed(batch))

            e = outs[self.output_index] if self.output_index is not None else outs[0]
            e = e.astype(np.float32)

            # IFIRでも pooling込みONNXなら [B,D] のはず。違ったら pooling が必要。
            if e.ndim != 2:
                raise RuntimeError(f"Model output is not [B, D]. Got shape={e.shape}. Need pooling.")

            if norm:
                e = e / (np.linalg.norm(e, axis=1, keepdims=True) + 1e-12)

            embs.append(e)

        return np.concatenate(embs, axis=0)

model = AimetMtebModel(sim=sim, tokenizer=tokenizer, seq_len=512, batch_size=32, normalize=True)

# 3) evaluate() で実行（ここがポイント：deprecatedのMTEBクラスを使わない）
#    tasks は Taskオブジェクトのリストで渡せます
results = evaluate(
    model=model,
    tasks=[task],
    output_folder="./mteb_ifir_scifact_aimet_results",
)

print(results)

AttributeError: 'AbsTaskRetrieval' object has no attribute 'metadata'

In [ ]:

import pkgutil, mteb.tasks.retrieval as r
mods = [m.name for m in pkgutil.walk_packages(r.__path__, r.__name__ + ".")]
print([m for m in mods if "scifact" in m.lower()])



[]


In [ ]:
# ----------------------------
# 7. Step5: Export（成果物を出す）
# ----------------------------
from pathlib import Path
Path("./out_ptq").mkdir(exist_ok=True)

sim.export(path="./out_ptq", filename_prefix="bge_w8a8_dynamo_on_ptq")  # docsのONNX export [1](https://quic.github.io/aimet-pages/releases/latest/techniques/ptq.html)

print("✅ Export completed")

Done. Exported to: /root/AIMET_ruri_v2/tool/out_ptq
QDQ model: /root/AIMET_ruri_v2/tool/out_ptq/ruri_quat_default.onnx
Encodings: /root/AIMET_ruri_v2/tool/out_ptq/ruri_quat_default.encodings
